In [ ]:
!python -m ensurepip --upgrade

In [ ]:
%pip install openai requests

# Test Nemotron-Parse

In [ ]:
# Download a sample image
import requests

url = "https://upload.wikimedia.org/wikipedia/commons/c/c3/LibreOffice_Writer_6.3.png"
image_path = "sample_image.png"
headers = {"User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36"}
r = requests.get(url, headers=headers)
r.raise_for_status()
with open(image_path, "wb") as f:
    f.write(r.content)
print(f"Downloaded to {image_path}")

from IPython.display import Image, display
display(Image(filename=image_path, width=400))

*Sample image: [LibreOffice Writer 6.3](https://commons.wikimedia.org/wiki/File:LibreOffice_Writer_6.3.png) by [Hatas](https://commons.wikimedia.org/wiki/User:Hatas), Wikimedia Commons, [CC BY-SA 4.0](https://creativecommons.org/licenses/by-sa/4.0/deed.en).*

In [ ]:
import base64
import os
from openai import OpenAI

IMAGE_MIME = {"png": "image/png", "jpg": "image/jpeg", "jpeg": "image/jpeg", "webp": "image/webp"}

def get_extension(filename):
    _, ext = os.path.splitext(filename)
    return ext[1:].lower()

def encode_image_base64(image_path: str) -> str:
    ext = get_extension(image_path)
    if ext not in IMAGE_MIME:
        raise ValueError(f"Unsupported image format: {ext}")
    with open(image_path, "rb") as f:
        b64 = base64.b64encode(f.read()).decode("utf-8")
    return f"data:{IMAGE_MIME[ext]};base64,{b64}"

client = OpenAI(
    base_url="http://localhost:8002/v1",
    api_key="not-needed",
)

try:
    print(f"Encoding {image_path} as base64...")
    image_url = encode_image_base64(image_path)
    completion = client.chat.completions.create(
        model="nvidia/nemotron-parse",
        tools=[{"type": "function", "function": {"name": "markdown_bbox"}}],
        messages=[
            {"role": "user", "content": [{"type": "image_url", "image_url": {"url": image_url}}]},
        ],
        temperature=0,
        max_tokens=256
    )
    print("Parse request: Success")
    raw = completion.model_dump() if hasattr(completion, "model_dump") else completion
    print("--- Raw completion (full API response) ---")
    import json
    print(json.dumps(raw, indent=2, default=str))
    if completion.choices and completion.choices[0].message.tool_calls:
        args = completion.choices[0].message.tool_calls[0].function.arguments
        out = json.loads(args)
        text = out.get("text", out) if isinstance(out, dict) else out
        print("--- Extracted text (from tool_calls) ---")
        print(text if isinstance(text, str) else text)
except Exception as e:
    print(f"Parse request: Failed — {e}")

# Test Embedding Model

In [ ]:
from openai import OpenAI

client = OpenAI(
    api_key="not-needed",
    base_url="http://localhost:8003/v1",
)

try:
    response = client.embeddings.create(
        input=["What is the civil caseload in South Dakota courts?"],
        model="nvidia/llama-3.2-nv-embedqa-1b-v2",
        encoding_format="float",
        extra_body={"input_type": "query"},
    )
    print("Embedding request: Success")
    print(response.data[0].embedding)
except Exception as e:
    print(f"Embedding request: Failed — {e}")

# Test VLM

In [ ]:
import requests
import os
import base64

query = "Describe the scene"

invoke_url = "http://localhost:8004/v1/chat/completions"

# Image-only support for this notebook
IMAGE_MIME = {
    "png": "image/png",
    "jpg": "image/jpeg",
    "jpeg": "image/jpeg",
    "webp": "image/webp",
}


def get_extension(filename):
    _, ext = os.path.splitext(filename)
    return ext[1:].lower()


def encode_image_base64(image_path: str) -> str:
    """Encode image file to base64 data URL."""
    ext = get_extension(image_path)
    if ext not in IMAGE_MIME:
        raise ValueError(f"Unsupported image format: {ext}. Use one of {list(IMAGE_MIME)}")
    with open(image_path, "rb") as f:
        b64 = base64.b64encode(f.read()).decode("utf-8")
    mime = IMAGE_MIME[ext]
    return f"data:{mime};base64,{b64}"


# Encode single image and build request
print(f"Encoding {image_path} as base64...")
image_url = encode_image_base64(image_path)

content = [
    {"type": "text", "text": query},
    {"type": "image_url", "image_url": {"url": image_url}},
]

headers = {
    "Content-Type": "application/json",
    "Accept": "application/json",
}


messages = [
    {"role": "system", "content": "/think"},
    {"role": "user", "content": content},
]

payload = {
    "max_tokens": 4096,
    "temperature": 1,
    "top_p": 1,
    "frequency_penalty": 0,
    "presence_penalty": 0,
    "messages": messages,
    "model": "nvidia/nemotron-nano-12b-v2-vl",
}

try:
    response = requests.post(invoke_url, headers=headers, json=payload)
    response.raise_for_status()
    print("VLM request: Success")
    try:
        print(response.json())
    except ValueError:
        # Response is not JSON (empty or other format)
        print("Response body (raw):", response.text or "(empty)")
except Exception as e:
    print(f"VLM request: Failed — {e}")